# 強化学習 入門ハンズオン 🧠🎮

このノートブックは **強化学習 (Reinforcement Learning, RL)** をゼロから手を動かして学ぶための教材です。
最終的に、このリポジトリ (Qubit) に実装されている **DPO (Direct Preference Optimization)** が
強化学習 / RLHF とどう繋がっているのかまで理解することを目指します。

## 学習の流れ

1. 強化学習とは何か（用語と全体像）
2. 多腕バンディット問題（探索と活用のトレードオフ）
3. マルコフ決定過程 (MDP) と価値の考え方
4. Q学習をグリッドワールドで実装
5. 方策勾配 (Policy Gradient) の考え方
6. RLHF と DPO — 言語モデルにおける強化学習
7. このリポジトリの `dpo_utils.py` を実際に動かす

> **前提知識**: Python の基本、numpy。後半で PyTorch を少し使います。
> **実行環境**: `pip install numpy torch matplotlib` があれば動きます。


## 0. セットアップ

まずは必要なライブラリを読み込みます。`matplotlib` が無い環境ではグラフ表示部分だけスキップされます。


In [ ]:
import numpy as np

np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False
    print("matplotlib が無いためグラフ表示はスキップします")

print("numpy version:", np.__version__)


## 1. 強化学習とは何か

強化学習は **「エージェントが環境と相互作用しながら、報酬を最大化する行動を試行錯誤で学ぶ」** 枠組みです。

```
        行動 (action)
   ┌─────────────────────┐
   │                     ▼
┌──────┐            ┌─────────┐
│ Agent│            │  環境    │
│エージェント│        │Environment│
└──────┘            └─────────┘
   ▲                     │
   └─────────────────────┘
     状態 (state) + 報酬 (reward)
```

### 主要な用語

| 用語 | 記号 | 意味 |
|------|------|------|
| 状態 | $s$ | 環境の今の状況 |
| 行動 | $a$ | エージェントが選ぶ手 |
| 報酬 | $r$ | 行動の結果として得る数値 |
| 方策 | $\pi(a\mid s)$ | 状態に対しどの行動を取るかの確率 |
| 価値 | $V(s),\ Q(s,a)$ | 将来得られる報酬の期待値 |
| 割引率 | $\gamma$ | 未来の報酬をどれだけ重視するか (0〜1) |

ゴールは **累積報酬（リターン）** $G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots$ を最大化する方策 $\pi$ を見つけることです。

教師あり学習との違いは「正解ラベルが無く、報酬という弱い信号だけで学ぶ」点です。


## 2. 多腕バンディット問題 — 探索と活用

強化学習で最も基本的な題材が **多腕バンディット (Multi-Armed Bandit)** です。

複数のスロットマシン（腕）があり、それぞれ当たる確率が違います。
どれが良いか分からない状態で、合計報酬を最大化するにはどうするか？

ここで重要なのが **探索 (exploration) と活用 (exploitation) のトレードオフ** です。
- 活用: 今まで一番良かった腕を引き続ける
- 探索: まだ試していない腕を試して、もっと良い腕を探す

代表的な戦略が **ε-greedy**: 確率 ε でランダムに探索、それ以外は今ベストの腕を選ぶ。


In [ ]:
class BanditEnv:
    '''各腕が異なる当たり確率を持つベルヌーイ・バンディット'''
    def __init__(self, probs):
        self.probs = np.array(probs)
        self.n_arms = len(probs)

    def pull(self, arm):
        # 当たれば報酬1、外れれば0
        return 1.0 if np.random.rand() < self.probs[arm] else 0.0


def epsilon_greedy_bandit(env, steps=2000, epsilon=0.1):
    Q = np.zeros(env.n_arms)       # 各腕の推定価値
    counts = np.zeros(env.n_arms)  # 各腕を引いた回数
    rewards = []

    for t in range(steps):
        if np.random.rand() < epsilon:
            arm = np.random.randint(env.n_arms)   # 探索
        else:
            arm = int(np.argmax(Q))               # 活用

        r = env.pull(arm)
        counts[arm] += 1
        # 推定価値をインクリメンタル平均で更新
        Q[arm] += (r - Q[arm]) / counts[arm]
        rewards.append(r)

    return Q, counts, np.array(rewards)


env = BanditEnv(probs=[0.2, 0.5, 0.75, 0.4])
Q, counts, rewards = epsilon_greedy_bandit(env, steps=2000, epsilon=0.1)

print("真の当たり確率 :", env.probs)
print("推定価値 Q     :", np.round(Q, 3))
print("各腕の選択回数 :", counts.astype(int))
print("平均報酬       :", rewards.mean().round(3))
print("最良の腕       :", int(np.argmax(Q)), "(真の最良:", int(np.argmax(env.probs)), ")")


### ε の影響を比べてみる

ε が大きいと探索しすぎ、小さいと活用に偏ります。累積報酬の伸びを比較します。


In [ ]:
if HAS_PLT:
    plt.figure(figsize=(8, 4))
    for eps in [0.0, 0.05, 0.1, 0.3]:
        np.random.seed(0)
        env = BanditEnv(probs=[0.2, 0.5, 0.75, 0.4])
        _, _, r = epsilon_greedy_bandit(env, steps=2000, epsilon=eps)
        plt.plot(np.cumsum(r), label=f"ε={eps}")
    plt.xlabel("step")
    plt.ylabel("累積報酬")
    plt.title("ε-greedy: 探索率による累積報酬の違い")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("matplotlib が無いためスキップ")


ε=0（純粋な活用）は最初に運悪く悪い腕に固定されると伸び悩みます。
適度な探索 (ε=0.05〜0.1) が長期的に有利になることが多いです。


## 3. マルコフ決定過程 (MDP)

バンディットは状態が1つだけの特殊ケースでした。
一般の強化学習では **状態が遷移** します。これを定式化したのが **マルコフ決定過程 (MDP)** です。

MDP は $(S, A, P, R, \gamma)$ の組で定義されます。
- $S$: 状態の集合
- $A$: 行動の集合
- $P(s' \mid s, a)$: 遷移確率
- $R(s, a)$: 報酬
- $\gamma$: 割引率

**ベルマン方程式** が価値の関係を表します:

$$Q(s,a) = R(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, \max_{a'} Q(s', a')$$

「今の行動の報酬 + 次の状態以降で得られる最善の価値」を表しています。これが次の Q 学習の土台です。


## 4. Q学習をグリッドワールドで実装

**グリッドワールド** はRL学習の定番です。エージェントがマス目を動いてゴールを目指します。

```
. . . . G    G = ゴール (+1)
. # # . .    # = 壁 (進入不可)
. . . # .    X = 落とし穴 (-1)
. # . . .
S # . . X    S = スタート
```

**Q学習** はモデルフリー（遷移確率を知らなくても良い）な学習法で、次の更新式で Q テーブルを更新します:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \big]$$

- $\alpha$: 学習率
- 角括弧の中は **TD誤差 (Temporal Difference error)** と呼ばれます


In [ ]:
class GridWorld:
    def __init__(self):
        self.rows, self.cols = 5, 5
        self.walls = {(1,1),(1,2),(2,3),(3,1),(4,1)}
        self.goal = (0, 4)
        self.pit = (4, 4)
        self.start = (4, 0)
        self.actions = [(-1,0),(1,0),(0,-1),(0,1)]  # 上,下,左,右
        self.action_names = ["↑","↓","←","→"]

    def reset(self):
        self.pos = self.start
        return self.pos

    def step(self, a):
        dr, dc = self.actions[a]
        nr, nc = self.pos[0]+dr, self.pos[1]+dc
        # 範囲外・壁なら動かない
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr,nc) not in self.walls:
            self.pos = (nr, nc)
        if self.pos == self.goal:
            return self.pos, 1.0, True
        if self.pos == self.pit:
            return self.pos, -1.0, True
        return self.pos, -0.01, False   # 1歩ごとに小さなペナルティ（早くゴールさせる）


def train_q_learning(env, episodes=1500, alpha=0.1, gamma=0.95, epsilon=0.2):
    Q = np.zeros((env.rows, env.cols, len(env.actions)))
    returns = []
    for ep in range(episodes):
        s = env.reset()
        done, total = False, 0.0
        for _ in range(100):  # 最大ステップ数
            if np.random.rand() < epsilon:
                a = np.random.randint(len(env.actions))
            else:
                a = int(np.argmax(Q[s[0], s[1]]))
            s2, r, done = env.step(a)
            # Q学習の更新式
            best_next = np.max(Q[s2[0], s2[1]])
            td_target = r + gamma * best_next * (not done)
            Q[s[0], s[1], a] += alpha * (td_target - Q[s[0], s[1], a])
            s = s2
            total += r
            if done:
                break
        returns.append(total)
    return Q, returns


env = GridWorld()
Q, returns = train_q_learning(env)
print("学習完了。最後の100エピソードの平均リターン:", np.mean(returns[-100:]).round(3))


### 学習した方策を可視化

各マスで Q 値が最大の行動（矢印）を表示します。スタートからゴールへの経路ができていれば成功です。


In [ ]:
env = GridWorld()
policy = np.full((env.rows, env.cols), " ", dtype=object)
for r in range(env.rows):
    for c in range(env.cols):
        if (r,c) in env.walls:
            policy[r,c] = "#"
        elif (r,c) == env.goal:
            policy[r,c] = "G"
        elif (r,c) == env.pit:
            policy[r,c] = "X"
        else:
            policy[r,c] = env.action_names[int(np.argmax(Q[r,c]))]

print("学習した方策（各マスの最適行動）:")
for row in policy:
    print(" ".join(row))

if HAS_PLT:
    plt.figure(figsize=(7,3))
    # 移動平均で学習曲線を滑らかに表示
    w = 50
    smooth = np.convolve(returns, np.ones(w)/w, mode="valid")
    plt.plot(smooth)
    plt.xlabel("episode")
    plt.ylabel("リターン (移動平均)")
    plt.title("Q学習の学習曲線")
    plt.grid(alpha=0.3)
    plt.show()


学習が進むにつれてリターンが上がり、エージェントが壁や落とし穴を避けてゴールへ向かう方策を獲得します。

ここまでが **価値ベース (value-based)** の強化学習です。次は **方策ベース (policy-based)** を見ます。


## 5. 方策勾配 (Policy Gradient) の考え方

Q学習は「価値」を学んでから行動を決めました。
**方策勾配法** は方策 $\pi_\theta(a\mid s)$ を直接パラメータ $\theta$ で表し、報酬が増える方向へ勾配上昇します。

REINFORCE の目的関数の勾配は:

$$\nabla_\theta J(\theta) = \mathbb{E}\big[ \nabla_\theta \log \pi_\theta(a\mid s)\, G_t \big]$$

直感的には **「良い結果(高いリターン)につながった行動の確率を上げる」** という更新です。

この「ログ確率 × 良し悪し」という形は、後で出てくる **DPO/RLHF と本質的に同じ構造** をしています。
ここではバンディット上でソフトマックス方策を勾配で学習する最小例を示します。


In [ ]:
def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


def reinforce_bandit(env, steps=3000, lr=0.1):
    theta = np.zeros(env.n_arms)   # 各腕の選好(preference)パラメータ
    rewards = []
    baseline = 0.0
    for t in range(steps):
        probs = softmax(theta)
        a = np.random.choice(env.n_arms, p=probs)   # 方策からサンプリング
        r = env.pull(a)
        baseline += (r - baseline) * 0.01           # 分散を下げるベースライン
        # ∇logπ = (one_hot(a) - probs)。報酬が baseline より良ければ確率↑
        grad = -probs
        grad[a] += 1.0
        theta += lr * (r - baseline) * grad
        rewards.append(r)
    return theta, np.array(rewards)


env = BanditEnv(probs=[0.2, 0.5, 0.75, 0.4])
theta, r = reinforce_bandit(env)
print("学習後の方策（各腕を選ぶ確率）:", np.round(softmax(theta), 3))
print("真の当たり確率               :", env.probs)
print("最良の腕に最大確率が付いたか :", int(np.argmax(softmax(theta))) == int(np.argmax(env.probs)))


## 6. RLHF と DPO — 言語モデルにおける強化学習

近年の大規模言語モデル (LLM) は **RLHF (Reinforcement Learning from Human Feedback)** で人間の好みに合わせて調整されます。

### RLHF の標準的な流れ
1. **報酬モデルの学習**: 人間が「どちらの回答が好きか」を比較したデータから報酬モデルを学習
2. **方策の最適化**: その報酬を最大化するよう LLM を強化学習 (PPO) で更新

ここでの LLM = エージェント、出力する文章 = 行動、報酬モデル = 報酬、という対応です。
つまり「文章生成」を逐次的な意思決定問題として強化学習で解いています。

### DPO (Direct Preference Optimization)
RLHF は報酬モデルの学習や PPO が複雑で不安定になりがちです。
**DPO** は報酬モデルを明示的に作らず、選好ペア (chosen / rejected) から直接方策を最適化します。

このリポジトリの `DPO_GUIDE.md` にある損失関数は:

$$\mathcal{L}_{DPO} = -\log \sigma\big(\beta\,(\log p_\theta(y_c\mid x) - \log p_\theta(y_r\mid x))\big)$$

- $y_c$: 好まれた (chosen) 回答, $y_r$: 好まれなかった (rejected) 回答
- $\beta$: 選好の強さを決める温度パラメータ

**ポイント**: 第5節の方策勾配 `∇logπ × 良し悪し` と見比べてください。
DPO も「chosen のlog確率を上げ、rejected のlog確率を下げる」更新で、
強化学習の発想をシンプルな教師あり損失に落とし込んだものだと分かります。


In [ ]:
# DPO 損失の挙動を最小コードで体感する（numpy）
def dpo_loss_numpy(logp_chosen, logp_rejected, beta=0.5):
    log_odds = beta * (logp_chosen - logp_rejected)
    return -np.log(1.0 / (1.0 + np.exp(-log_odds)))  # -log(sigmoid(log_odds))

# chosen の log確率が rejected より高いほど損失は下がる
print("chosen >> rejected の場合の損失:", round(dpo_loss_numpy(-2.0, -8.0), 4))
print("chosen ≈  rejected の場合の損失:", round(dpo_loss_numpy(-5.0, -5.0), 4))
print("chosen << rejected の場合の損失:", round(dpo_loss_numpy(-8.0, -2.0), 4))

if HAS_PLT:
    diff = np.linspace(-6, 6, 100)  # log p_chosen - log p_rejected
    for beta in [0.1, 0.5, 1.0]:
        plt.plot(diff, [dpo_loss_numpy(d, 0, beta) for d in diff], label=f"β={beta}")
    plt.xlabel("log p(chosen) - log p(rejected)")
    plt.ylabel("DPO loss")
    plt.title("DPO損失: chosenが優位なほど損失が下がる")
    plt.axvline(0, color="gray", ls="--", alpha=0.5)
    plt.legend(); plt.grid(alpha=0.3); plt.show()


## 7. このリポジトリの DPO 実装を動かす

`dpo_utils.py` には本物の DPO 損失計算 `compute_dpo_loss` が実装されています（PyTorch）。
ダミーのロジットを使って実際に呼び出し、`compute_dpo_metrics` で精度を確認します。

> このセルは PyTorch が必要です。リポジトリのルートで実行してください。


In [ ]:
import sys, os
# notebooks/ から1つ上（リポジトリのルート）を import パスに追加
sys.path.insert(0, os.path.abspath(".."))

import torch
from dpo_utils import compute_dpo_loss, compute_dpo_metrics, create_preference_examples

torch.manual_seed(0)

batch, seq_len, vocab = 2, 6, 50

# chosen 側はわざと正解ラベルに高いロジットを与え「良い回答」を模擬する
labels_chosen   = torch.randint(0, vocab, (batch, seq_len))
labels_rejected = torch.randint(0, vocab, (batch, seq_len))

logits_chosen   = torch.randn(batch, seq_len, vocab)
logits_rejected = torch.randn(batch, seq_len, vocab)
# chosen の正解トークンのロジットを底上げ → モデルが chosen を好む状態を作る。
# compute_dpo_loss は次トークン予測のため1つシフトするので、
# 位置 t のロジットは labels[t+1] を予測する点に合わせて底上げする。
for b in range(batch):
    for t in range(seq_len - 1):
        logits_chosen[b, t, labels_chosen[b, t + 1]] += 5.0

loss, logp_c, logp_r = compute_dpo_loss(
    logits_chosen, logits_rejected, labels_chosen, labels_rejected, beta=0.5
)
metrics = compute_dpo_metrics(logp_c, logp_r)

print("DPO loss        :", round(loss.item(), 4))
print("log p(chosen)   :", logp_c.detach().numpy().round(3))
print("log p(rejected) :", logp_r.detach().numpy().round(3))
print("preference acc  :", round(metrics['accuracy'], 3), "(chosen>rejected の割合)")
print("avg log ratio   :", round(metrics['avg_log_ratio'], 3))


### 用意されている選好データの例

`create_preference_examples()` で、このリポジトリに同梱された日本語の選好ペアを確認できます。
これらが DPO 学習の入力（chosen / rejected）になります。


In [ ]:
examples = create_preference_examples()
for i, ex in enumerate(examples, 1):
    print(f"[{i}] prompt : {ex['prompt']}")
    print(f"    chosen  : {ex['chosen']}")
    print(f"    rejected: {ex['rejected']}")
    print()


## まとめ

- **強化学習** はエージェントが報酬を頼りに試行錯誤で行動を学ぶ枠組み
- **多腕バンディット** で探索と活用のトレードオフ（ε-greedy）を学んだ
- **Q学習** で状態遷移のある問題を価値ベースで解いた（グリッドワールド）
- **方策勾配 (REINFORCE)** で方策を直接最適化した。更新は「log確率 × 良し悪し」
- **RLHF / DPO** は LLM を人間の好みに合わせる強化学習。DPO はそれを簡潔な損失に落とし込んだもの
- このリポジトリの `dpo_utils.py` の `compute_dpo_loss` を実際に動かして確認した

### 次のステップ
- `DPO_GUIDE.md` を読み、`train_dpo.py` / `train_combined_dpo.py` を試す
- グリッドワールドを難しくして Q学習のハイパーパラメータ (α, γ, ε) の影響を観察する
- 方策勾配を PyTorch で実装し、ニューラルネット方策に拡張する

📚 **参考文献**
- Sutton & Barto, *Reinforcement Learning: An Introduction*
- Rafailov et al., *Direct Preference Optimization: Your Language Model is Secretly a Reward Model* (2023)
